# IMDB Movie Reviews Sentiment Analysis and Model Deployment with Flask

## What is Sentiment?

Sentiment analysis is the interpretation and classification of emotions within text data using text analysis techniques.
- Positive
- Neutral
- Negative



In [ ]:
import pandas as pd
import numpy as np

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [ ]:
import re

def clean_text(text):
    text = text.lower()
    text = re.sub(r'http\S+', '', text)
    text = re.sub(r'\d+', '', text)
    text = re.sub(r'[^\w\s]', '', text)
    return text

In [ ]:
dataset = pd.read_csv('imdb_reviews.txt', sep = '\t', header = None)

In [ ]:
dataset.columns = ['reviews', 'sentiment']

In [ ]:
dataset.head()

,reviews,sentiment
0,"A very, very, very slow-moving, aimless movie ...",0
1,Not sure who was more lost - the flat characte...,0
2,Attempting artiness with black & white and cle...,0
3,Very little music or anything to speak of.,0
4,The best scene in the movie was when Gerardo i...,1


In [ ]:
dataset.isnull().sum()

,0
reviews,0
sentiment,0


In [ ]:
dataset.dropna(inplace=True)

In [ ]:
x = dataset['reviews']
y = dataset['sentiment']

In [ ]:
x = x.apply(clean_text)

In [ ]:
print(x)

0      a very very very slowmoving aimless movie abou...
1      not sure who was more lost  the flat character...
2      attempting artiness with black  white and clev...
3            very little music or anything to speak of  
4      the best scene in the movie was when gerardo i...
                             ...                        
743    i just got bored watching jessice lange take h...
744    unfortunately any virtue in this films product...
745                       in a word it is embarrassing  
746                                  exceptionally bad  
747    all in all its an insult to ones intelligence ...
Name: reviews, Length: 748, dtype: object


In [ ]:
print(y)

0      0
1      0
2      0
3      0
4      1
      ..
743    0
744    0
745    0
746    0
747    0
Name: sentiment, Length: 748, dtype: int64


In [ ]:
from sklearn.model_selection import train_test_split
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42, stratify = y)

In [ ]:
model = Pipeline([
    ('tfidf', TfidfVectorizer(max_df=0.8, ngram_range=(1,2))),
    ('clf', LogisticRegression(max_iter=1000))
])

In [ ]:
model.fit(x_train, y_train)

Pipeline(steps=[('tfidf', TfidfVectorizer(max_df=0.8, ngram_range=(1, 2))),
                ('clf', LogisticRegression(max_iter=1000))])

In [ ]:
y_pred = model.predict(x_test)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.77      0.73      0.75        73
           1       0.75      0.79      0.77        77

    accuracy                           0.76       150
   macro avg       0.76      0.76      0.76       150
weighted avg       0.76      0.76      0.76       150



In [ ]:
examples = ["This movie was amazing!", "I hated this film, so boring."]
print(model.predict(examples))

[1 0]


In [ ]:
examples = ["This movie was amazing!", "I hated this film, so boring."]
preds = model.predict(examples)

for text, label in zip(examples, preds):
    sentiment = "Positive" if label == 1 else "Negative"
    print(f"{text} → {sentiment}")

This movie was amazing! → Positive
I hated this film, so boring. → Negative


In [ ]:
examples = ["This movie was not amazing!", "I hated this film, so boring."]
print(model.predict(examples))

[0 0]


In [ ]:
from transformers import pipeline

distilbert_model = pipeline("sentiment-analysis", model="distilbert-base-uncased-finetuned-sst-2-english")

examples = [
    "This movie was amazing!",
    "I hated this film, so boring.",
    "It wasn't amazing.",
    "It was not amazing."
]

results = distilbert_model(examples)

for text, res in zip(examples, results):
    print(f"Text: {text}\nPrediction: {res}\n")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

Device set to use cpu


Text: This movie was amazing!
Prediction: {'label': 'POSITIVE', 'score': 0.9998800754547119}

Text: I hated this film, so boring.
Prediction: {'label': 'NEGATIVE', 'score': 0.999754011631012}

Text: It wasn't amazing.
Prediction: {'label': 'NEGATIVE', 'score': 0.9996960163116455}

Text: It was not amazing.
Prediction: {'label': 'NEGATIVE', 'score': 0.9997765421867371}

